# Proiect - Agent care asista utilizatorul in procesul de recomandare a unor produse

Pasul 1: Incarc datele. Voi folosi setul de date UCI Online Retail, care contine 541909 de instante. Avem 6 features: Description, Quantity, InoviceDate, UnitPrice, CustomerID, Country si 2 variabile de tip ID: InoviceNo si StockCode.

https://archive.ics.uci.edu/dataset/352/online+retail


Aceasta este descrierea de pe site:

This is a transactional data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.The company mainly sells unique all-occasion gifts. Many customers of the company are wholesalers.


*   InvoiceNo: Invoice number. Nominal, a 6-digit integral number uniquely assigned to each transaction. If this code starts with letter 'c', it indicates a cancellation.

*   StockCode: Product (item) code. Nominal, a 5-digit integral number uniquely assigned to each distinct product.

*   Description: Product (item) name. Nominal.

*   Quantity: The quantities of each product (item) per transaction. Numeric.


*   InvoiceDate: Invoice Date and time. Numeric, the day and time when each transaction was generated.


*   UnitPrice: Unit price. Numeric, Product price per unit in sterling.

*   CustomerID: Customer number. Nominal, a 5-digit integral number uniquely assigned to each customer.
*   Country: Country name. Nominal, the name of the country where each customer resides.













In [1]:
import pandas as pd
import requests
import zipfile
from io import BytesIO

url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

response = requests.get(url, timeout=120)
response.raise_for_status()

with zipfile.ZipFile(BytesIO(response.content)) as archive:
    excel_file = next(
        name for name in archive.namelist()
        if name.lower().endswith(".xlsx")
    )

    with archive.open(excel_file) as file:
        df = pd.read_excel(file)

print("Dataset incarcat")
print("Numar de tranzactii (linii) si features (coloane):", df.shape)


Dataset incarcat
Numar de tranzactii (linii) si features (coloane): (541909, 8)


Afisez cateva informatii

In [2]:
print("coloanele datasetului:")
print(df.columns.tolist())

print("\nprimele 5 inregistrari:")
display(df.head())

print("\ninfo:")
df.info()


coloanele datasetului:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

primele 5 inregistrari:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom



info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


Pasul 2: prelucrarea datelor

Observam ca primele 5 afisari au acelasi InvoiceID, ceea ce sugereaza ca produsele cumparate apartin aceleiasi facturi. Este mentionat in descrierea setului ca id-urile comenzilor care incep cu "c" sunt comenzi anulate, asa ca m-am gandit sa le elimin. De asemenea, numeroase instante au valori lipsa pentru diferite atribute, asa ca am eliminat si acele inregistrari, precum si cele unde pretul sau cantitatea era nr negativ.

In [8]:
#cate inregistrari au val lipsa
print("valori lipsa pe coloane:")
print(df.isnull().sum())

#caut cate comenzi incep cu litera c
print("\nnumar de comenzi care au id-ul care incepe cu C:")
print(df["InvoiceNo"].astype(str).str.startswith("C").sum())

#caut cate inregistrari au o valoare negativa sau 0
print("\ncantitati negative sau zero:")
print((df["Quantity"] <= 0).sum())

#caut inregistrarile unde pretul e negativ sau 0
print("\npreturi negative sau zero:")
print((df["UnitPrice"] <= 0).sum())

#caut inregistrari cu valori nule la atribute
print("\nclienti cu val lipsa:")
print(df["CustomerID"].nunique())


valori lipsa pe coloane:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

numar de comenzi care au id-ul care incepe cu C:
9288

cantitati negative sau zero:
10624

preturi negative sau zero:
2517

clienti cu val lipsa:
4372


In [9]:
#creez o copie pentru setul de date original
df_clean = df.copy()

#elimin inregistrarile fara client sau descriere
df_clean = df_clean.dropna(
    subset=["CustomerID", "Description"]
)

#elimin inregistrarile cu comanda anulata
df_clean = df_clean[
    ~df_clean["InvoiceNo"].astype(str).str.startswith("C")
]

#elimin inregistrarile cu cantitati sau preturi negative
df_clean = df_clean[
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0)
]

print("numar randuri initial:", len(df))
print("numar randuri dupa preprocesare:", len(df_clean))

numar randuri initial: 541909
numar randuri dupa preprocesare: 397884


Pasul 3: Incep sa creez functiile agentului

Prima functie a agentului:
Atunci cand un utilizator intreaba "Ce a comandat clientul X in trecut?", el sa raspunda cautand comenzile clientului in dataset.

Creez functia get_order_history(customer_id). Aceasta cauta prin intermediul id-ului clientului toate inregistrarile asociate lui, le grupeaza dupa facturi ca sa stie ce a cumparat la fiecare comanda data si returneaza informatiile gasite.

In [10]:
def get_order_history(customer_id):
    #selecteaza toate produsele cumparate de clientul cu id-ul X
    customer_orders = df_clean[df_clean["CustomerID"] == float(customer_id)].copy()

    #verifica daca gaseste ceva
    if customer_orders.empty:
        return "Nu exista comenzi pentru acest client."

    #selectez pentru partea de istoric doar atributele relevante
    history = customer_orders[
        [
            "InvoiceNo",
            "InvoiceDate",
            "StockCode",
            "Description",
            "Quantity",
            "UnitPrice"
        ]
    ]

    #sortez cronologic comenzile
    history = history.sort_values("InvoiceDate")

    #returnez cu indexul tabelului resetat dupa ce am ordonat cronologic
    return history.reset_index(drop=True)


Important de mentionat:
Am comparat ID-ul clientului primit cu valorile din coloana CustomerID. Pentru comparare, am convertit parametrul la float deoarece in dataset ID-urile sunt stocate ca numere zecimale, de exemplu 17850.0

Testez functia pentru un client din setul de date, respectiv pentru un client inexistent.

In [14]:
history = get_order_history(17850)
display(history.head(10))

,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice
0,536365,2010-12-01 08:26:00,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55
1,536365,2010-12-01 08:26:00,71053,WHITE METAL LANTERN,6,3.39
2,536365,2010-12-01 08:26:00,84406B,CREAM CUPID HEARTS COAT HANGER,8,2.75
3,536365,2010-12-01 08:26:00,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39
4,536365,2010-12-01 08:26:00,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,3.39
5,536365,2010-12-01 08:26:00,22752,SET 7 BABUSHKA NESTING BOXES,2,7.65
6,536365,2010-12-01 08:26:00,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,4.25
7,536366,2010-12-01 08:28:00,22633,HAND WARMER UNION JACK,6,1.85
8,536366,2010-12-01 08:28:00,22632,HAND WARMER RED POLKA DOT,6,1.85
9,536372,2010-12-01 09:01:00,22633,HAND WARMER UNION JACK,6,1.85


In [24]:
print(get_order_history(999999))

Nu exista comenzi pentru acest client.


Clientul 17850, ales random, a dat 2 comenzi in 2 zile consecutive. Cumpararea primului produs de 2 ori in 2 zile consecutive nu ajuta neaparat ideea de predictie. Nu garanteaza ca persoana va cumpara si urmatoarea zi. Poate fi o comanda mai mare impartita in 2 zile.

Urmeaza partea in care trebuie identificate produsele cumparate recurent. Functia va ajuta recomandarile agentului.
Aflu:
* ce produse cumpara clientul X;
* de cate ori le-a cumparat;
* cand le-a cumparat prima si ultima data;
* la ce interval mediu repeta cumpararea.

In [25]:
def get_reorder_recommendations(customer_id):
    history = get_order_history(customer_id)

    if isinstance(history, str):
        return history

    #pastrez doar data, fara ora din setul de date
    history["order_date"] = (history["InvoiceDate"].dt.normalize())

    #am agregat cantitatile cumparate in aceeasi zi
    daily_purchases = (
        history.groupby(
            ["StockCode", "Description", "order_date"]
        )
        .agg(
            daily_quantity=("Quantity", "sum")
        )
        .reset_index()
    )

    #calculam statisticile: prima zi de cumparare, ultima, cantitati zilnice
    summary = (
        daily_purchases.groupby(["StockCode", "Description"])
        .agg(
            purchase_days=("order_date", "nunique"),
            total_quantity=("daily_quantity", "sum"),
            first_order=("order_date", "min"),
            last_order=("order_date", "max")
        )
        .reset_index()
    )

    #facem media de zile intre comenzi
    summary["days_between_first_last"] = (
        summary["last_order"] - summary["first_order"]
    ).dt.days

    summary["avg_days_between_orders"] = (
        summary["days_between_first_last"]
        / (summary["purchase_days"] - 1).clip(lower=1)
    )

    #pastram produsele cumparate la interval mai mare de 3 zile
    #produsele cumparate in zile consecutive nu se iau in calcul
    recommendations = summary[
        summary["purchase_days"] >= 3
    ].copy()

    #afisez produsele cumparate mai des
    recommendations = recommendations.sort_values(
        ["purchase_days", "last_order"],
        ascending=[False, False]
    )

    return recommendations.reset_index(drop=True)

#apelez functia
recommendations = get_reorder_recommendations(14911)

recommendations.head(10)


,StockCode,Description,purchase_days,total_quantity,first_order,last_order,days_between_first_last,avg_days_between_orders
0,C2,CARRIAGE,71,84,2010-12-01,2011-12-05,369,5.271429
1,22423,REGENCY CAKESTAND 3 TIER,46,342,2010-12-03,2011-12-07,369,8.200000
2,22699,ROSES REGENCY TEACUP AND SAUCER,31,465,2010-12-06,2011-12-08,367,12.233333
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,31,770,2010-12-16,2011-12-07,356,11.866667
4,22697,GREEN REGENCY TEACUP AND SAUCER,27,369,2010-12-06,2011-12-08,367,14.115385
5,21790,VINTAGE SNAP CARDS,25,804,2010-12-23,2011-12-07,349,14.541667
6,22698,PINK REGENCY TEACUP AND SAUCER,24,327,2011-05-23,2011-12-08,199,8.652174
7,22138,BAKING SET 9 PIECE RETROSPOT,24,147,2010-12-23,2011-12-07,349,15.173913
8,21212,PACK OF 72 RETROSPOT CAKE CASES,23,672,2010-12-09,2011-12-07,363,16.500000
9,23245,SET OF 3 REGENCY CAKE TINS,23,132,2011-06-09,2011-11-28,172,7.818182


Acum am informatii legate de istoricul de cumparare: Clientul a cumparat produsul 22423 in 46 de zile distincte, iar intervalul mediu dintre comenzi este 8,2 zile.

Acum voi face o functie pentru a obtine data cand a fost data ultima comanda a unui produs, pentru a putea prezice cand urmeaza urmatoarea.

Aleg ca referinta data 9 decembrie 2011, fiind ultima data din setul de date.

In [26]:
#9 dec 2011
reference_date = df_clean["InvoiceDate"].max().normalize()

recommendations["days_since_last_purchase"] = (reference_date - recommendations["last_order"]).dt.days

recommendations["estimated_days_until_reorder"] = (
    recommendations["avg_days_between_orders"]
    - recommendations["days_since_last_purchase"]
)

recommendations["reorder_status"] = recommendations[
    "estimated_days_until_reorder"
].apply(
    lambda days: (
        "Posibil de reaprovizionat"
        if days <= 0
        else "Inca nu trebuie aprovizionat"
    )
)

recommendations[
    [
        "StockCode",
        "Description",
        "purchase_days",
        "avg_days_between_orders",
        "last_order",
        "days_since_last_purchase",
        "estimated_days_until_reorder",
        "reorder_status"
    ]
].head(10)


,StockCode,Description,purchase_days,avg_days_between_orders,last_order,days_since_last_purchase,estimated_days_until_reorder,reorder_status
0,C2,CARRIAGE,71,5.271429,2011-12-05,4,1.271429,Inca nu trebuie aprovizionat
1,22423,REGENCY CAKESTAND 3 TIER,46,8.200000,2011-12-07,2,6.200000,Inca nu trebuie aprovizionat
2,22699,ROSES REGENCY TEACUP AND SAUCER,31,12.233333,2011-12-08,1,11.233333,Inca nu trebuie aprovizionat
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,31,11.866667,2011-12-07,2,9.866667,Inca nu trebuie aprovizionat
4,22697,GREEN REGENCY TEACUP AND SAUCER,27,14.115385,2011-12-08,1,13.115385,Inca nu trebuie aprovizionat
5,21790,VINTAGE SNAP CARDS,25,14.541667,2011-12-07,2,12.541667,Inca nu trebuie aprovizionat
6,22698,PINK REGENCY TEACUP AND SAUCER,24,8.652174,2011-12-08,1,7.652174,Inca nu trebuie aprovizionat
7,22138,BAKING SET 9 PIECE RETROSPOT,24,15.173913,2011-12-07,2,13.173913,Inca nu trebuie aprovizionat
8,21212,PACK OF 72 RETROSPOT CAKE CASES,23,16.500000,2011-12-07,2,14.500000,Inca nu trebuie aprovizionat
9,23245,SET OF 3 REGENCY CAKE TINS,23,7.818182,2011-11-28,11,-3.181818,Posibil de reaprovizionat


Exemplu: produsul 23245 — SET OF 3 REGENCY CAKE TINS: clientul l-a cumpărat în 23 de zile distincte, iar de la ultima cumpărare au trecut 11 zile. Intervalul mediu este de aproximativ 7,8 zile, deci produsul a depășit intervalul estimat cu aproximativ 3,2 zile.

Acum voi adauga stocul si cantitatea sugerata manual, deoarece nu am aceste date in setul de date. Asistentul va putea recomanda un produs doar daca este scadent pentru reaprovizionare si exista stoc suficient.

In [27]:
#stergem codurile care incep cu C
recommendations = recommendations[
    ~recommendations["StockCode"].astype(str).str.startswith("C")
].copy()

#generam random stocul
import random
random.seed(42)

recommendations["stock_available"] = [
    random.randint(0, 100)
    for _ in range(len(recommendations))
]

#cam cat din produsul X este cumparat in medie la o comanda
recommendations["avg_quantity_per_purchase_day"] = (
    recommendations["total_quantity"]
    / recommendations["purchase_days"]
)

#va sugera reaprovizionarea cu cantitatea comandata de obicei
recommendations["suggested_quantity"] = (
    recommendations["avg_quantity_per_purchase_day"]
    .round()
    .astype(int)
    .clip(lower=1)
)

#decizia in functie de interval si stoc
recommendations["decision"] = recommendations.apply(
    lambda row: (
        "Propune reaprovizionarea"
        if (
            row["estimated_days_until_reorder"] <= 0
            and row["stock_available"] >= row["suggested_quantity"]
        )
        else "Nu propune momentan"
    ),
    axis=1
)

recommendations[
    [
        "StockCode",
        "Description",
        "estimated_days_until_reorder",
        "stock_available",
        "suggested_quantity",
        "decision"
    ]
].head(15)


,StockCode,Description,estimated_days_until_reorder,stock_available,suggested_quantity,decision
1,22423,REGENCY CAKESTAND 3 TIER,6.200000,81,7,Nu propune momentan
2,22699,ROSES REGENCY TEACUP AND SAUCER,11.233333,14,15,Nu propune momentan
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,9.866667,3,25,Nu propune momentan
4,22697,GREEN REGENCY TEACUP AND SAUCER,13.115385,94,14,Nu propune momentan
5,21790,VINTAGE SNAP CARDS,12.541667,35,32,Nu propune momentan
6,22698,PINK REGENCY TEACUP AND SAUCER,7.652174,31,14,Nu propune momentan
7,22138,BAKING SET 9 PIECE RETROSPOT,13.173913,28,6,Nu propune momentan
8,21212,PACK OF 72 RETROSPOT CAKE CASES,14.500000,17,29,Nu propune momentan
9,23245,SET OF 3 REGENCY CAKE TINS,-3.181818,94,6,Propune reaprovizionarea
10,22960,JAM MAKING SET WITH JARS,15.761905,13,13,Nu propune momentan


23245: au trecut cu aproximativ 3,2 zile peste intervalul mediu; stocul este 94, iar cantitatea sugerata este 6, deci propune reaprovizionarea.



22699: intervalul mediu nu a fost depasit si stocul de 14 este chiar sub cantitatea sugerata de 15, deci nu propune momentan.

Pasul 4: Conectarea LLM-ului


La conectarea LLM-ului, voi imparti codul in 2 componente: una care sa abordeze cerinta clientului get_customer_summary(customer_id) si una care sa returneze produsele care sunt in stoc si trebuie aprovizionate get_reorder_recommendations(customer_id)

Incep cu transcrierea datelor intr-un format compatibil pentru un LLM, cum ar fi un dictionar Python.

In [28]:
def get_reorder_recommendations_for_agent(customer_id, top_n=5):
    #apeleaza functia de obtinere a recomandarilor pt clientul X
    result = get_reorder_recommendations(customer_id)

    if isinstance(result, str):
        return []

    #sterg codurile cu C
    result = result[
        ~result["StockCode"].astype(str).str.startswith("C")
    ].copy()

    #iau iar ca data de ref 9 dec 2011
    reference_date = df_clean["InvoiceDate"].max().normalize()

    result["days_since_last_purchase"] = (
        reference_date - result["last_order"]
    ).dt.days

    result["estimated_days_until_reorder"] = (
        result["avg_days_between_orders"]
        - result["days_since_last_purchase"]
    )

    #aproximeaza cantitatea sugerata pe baza istoricului
    result["suggested_quantity"] = (
        result["total_quantity"] / result["purchase_days"]
    ).round().astype(int).clip(lower=1)

    #simuleaza stocul
    rng = random.Random(42)

    result["stock_available"] = [
        rng.randint(0, 100) for _ in range(len(result))
    ]

    #selecteaza doar produsele care au atins media zilelor si ar trebui recumparate
    eligible = result[
        (result["purchase_days"] >= 3)
        & (result["estimated_days_until_reorder"] <= 0)
        & (
            result["stock_available"]
            >= result["suggested_quantity"]
        )
    ].copy()

    #pune prioritar produsele care au ajuns la valoarea medie recent
    eligible = eligible.sort_values(
        "estimated_days_until_reorder",
        ascending=False
    )

    columns = [
        "StockCode",
        "Description",
        "estimated_days_until_reorder",
        "stock_available",
        "suggested_quantity"
    ]

    return eligible[columns].head(top_n).to_dict(
        orient="records"
    )


get_reorder_recommendations_for_agent(14911)


[{'StockCode': 23169,
  'Description': 'CLASSIC GLASS COOKIE JAR',
  'estimated_days_until_reorder': -0.1999999999999993,
  'stock_available': 68,
  'suggested_quantity': 6},
 {'StockCode': 47566,
  'Description': 'PARTY BUNTING',
  'estimated_days_until_reorder': -1.0,
  'stock_available': 44,
  'suggested_quantity': 9},
 {'StockCode': 22228,
  'Description': 'BUNNY WOODEN PAINTED WITH BIRD ',
  'estimated_days_until_reorder': -1.2000000000000028,
  'stock_available': 70,
  'suggested_quantity': 18},
 {'StockCode': 21977,
  'Description': 'PACK OF 60 PINK PAISLEY CAKE CASES',
  'estimated_days_until_reorder': -1.2222222222222214,
  'stock_available': 47,
  'suggested_quantity': 26},
 {'StockCode': 21428,
  'Description': 'SET3 BOOK BOX GREEN GINGHAM FLOWER ',
  'estimated_days_until_reorder': -1.5999999999999996,
  'stock_available': 99,
  'suggested_quantity': 6}]

Avem foarte multe produse recomandate, multe pentru care ultima comanda a fost realizata acum foarte mult timp.

Valorile negative indica faptul ca intervalul estimat a fost depasit.

-1 - interval depasit cu o zi

-30 - interval depasit cu 30 de zile.

Ordonez descrescator dupa estimated_days_until_reorder, punand pe primul loc produsele care abia au ajuns la valoarea intervalului, peste produsele care au depasit cu mult intervalul.

Pasul 6: Interpretarea conversatiei

Pentru componenta conversationala, voi folosi un model open-source in colab. Acesta va interpreta conversatia cu clientul si va returna informatiile extrase despre produs, client, etc.

In [20]:
import torch

if torch.cuda.is_available():
    print("GPU disponibil:", torch.cuda.get_device_name(0))
else:
    print("Nu avem GPU disponibil momentan.")

GPU disponibil: Tesla T4


Voi incarca modelul Qwen2.5-1.5B-Instruct folosind biblioteca Hugging Face Transformers. Modelul are aproximativ 1,5 miliarde de parametri si poate genera răspunsuri conversaționale.

https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct?utm_source=chatgpt.com

Instalez biblioteca:

In [21]:
!pip install -q -U transformers accelerate

Incarc modelul:

In [30]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Modelul a fost incarcat!")
print("Dispozitiv:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelul a fost incarcat!
Dispozitiv: cuda:0


Conectez modelul incarcat la functia python

Pentru inceput, construiesc functia agentului

In [31]:
import re

def ask_reordering_agent(user_message):
    #extrag id-ul clientului din mesajul trimis de user
    match = re.search(r"\b\d{5}\b", user_message)

    if not match:
        return (
            "Te rog sa specifici ID-ul clientului, "
            "de exemplu 14911."
        )

    customer_id = int(match.group())

    #cauta clientul specificat in set
    customer_exists = (
        df_clean["CustomerID"] == float(customer_id)
    ).any()

    if not customer_exists:
        return (
            f"Nu am gasit clientul {customer_id} "
            "in dataset. Nu pot genera recomandari."
        )

    #apeleaza functia de recomandare
    recommendations = get_reorder_recommendations_for_agent(
        customer_id
    )

    if not recommendations:
        return (
            f"Nu exista produse eligibile pentru "
            f"reaprovizionare pentru clientul {customer_id}."
        )

    #formuleaza raspunsul pe baza info primite din functie
    lines = [
        f"Recomandari pentru clientul {customer_id}:"
    ]

    for i, item in enumerate(recommendations, start=1):
        lines.append(
            f"{i}. {item['Description']} "
            f"(cod: {item['StockCode']}) — "
            f"cantitate propusa: {item['suggested_quantity']} "
            f"bucati; stoc simulat: {item['stock_available']}."
        )

    lines.append(
        "\nAcestea sunt propuneri, nu comenzi plasate. "
        "Stocurile sunt simulate."
    )

    return "\n".join(lines)

Testez functionalitatea agentului in 3 situatii:

Testul 1: agentul obtine corect recomandari

Testul 2: cere ID-ul clientului, nu il ghiceste

Testul 3: cauta un client care nu exista, urmaresc sa nu inventeze istoric despre el

In [32]:
print("TEST 1")
print(ask_reordering_agent(
    "Ce produse ar trebui sa reaprovizionez pentru clientul 14911?"
))

print("\n" + "=" * 60 + "\n")

print("TEST 2")
print(ask_reordering_agent(
    "Ce produse ar trebui sa reaprovizionez?"
))

print("\n" + "=" * 60 + "\n")

print("TEST 3")
print(ask_reordering_agent(
    "Verifica reaprovizionarea pentru clientul 99999."
))

TEST 1
Recomandari pentru clientul 14911:
1. CLASSIC GLASS COOKIE JAR (cod: 23169) — cantitate propusa: 6 bucati; stoc simulat: 68.
2. PARTY BUNTING (cod: 47566) — cantitate propusa: 9 bucati; stoc simulat: 44.
3. BUNNY WOODEN PAINTED WITH BIRD  (cod: 22228) — cantitate propusa: 18 bucati; stoc simulat: 70.
4. PACK OF 60 PINK PAISLEY CAKE CASES (cod: 21977) — cantitate propusa: 26 bucati; stoc simulat: 47.
5. SET3 BOOK BOX GREEN GINGHAM FLOWER  (cod: 21428) — cantitate propusa: 6 bucati; stoc simulat: 99.

Acestea sunt propuneri, nu comenzi plasate. Stocurile sunt simulate.


TEST 2
Te rog sa specifici ID-ul clientului, de exemplu 14911.


TEST 3
Nu am gasit clientul 99999 in dataset. Nu pot genera recomandari.


Acum verific rezultatele, sa vad daca LLM-ul a raspuns corect

In [33]:
def inspect_recommendations(customer_id):
    results = get_reorder_recommendations_for_agent(customer_id)

    if not results:
        print(f"Clientul {customer_id}: nu exista recomandari.")
        return

    print(f"Client: {customer_id}")
    print(f"Numar recomandari: {len(results)}\n")

    for item in results:
        print(
            f"- {item['Description']}: "
            f"{item['suggested_quantity']} bucati propuse; "
            f"stoc simulat: {item['stock_available']}."
        )


inspect_recommendations(14911)

Client: 14911
Numar recomandari: 5

- CLASSIC GLASS COOKIE JAR: 6 bucati propuse; stoc simulat: 68.
- PARTY BUNTING: 9 bucati propuse; stoc simulat: 44.
- BUNNY WOODEN PAINTED WITH BIRD : 18 bucati propuse; stoc simulat: 70.
- PACK OF 60 PINK PAISLEY CAKE CASES: 26 bucati propuse; stoc simulat: 47.
- SET3 BOOK BOX GREEN GINGHAM FLOWER : 6 bucati propuse; stoc simulat: 99.


Pasul 7: Adaug tool-urile agentului

Incep cu istoricul comenzilor:

In [34]:
def tool_get_order_history(customer_id):
    customer_exists = (
        df_clean["CustomerID"] == float(customer_id)
    ).any()

    if not customer_exists:
        return {
            "success": False,
            "message": (
                f"Clientul {customer_id} nu exista in dataset."
            )
        }

    history = get_order_history(customer_id)

    if isinstance(history, str):
        return {
            "success": False,
            "message": history
        }

    history = history.tail(10).copy()
    history["InvoiceDate"] = history["InvoiceDate"].astype(str)

    return {
        "success": True,
        "customer_id": int(customer_id),
        "orders": history.to_dict(orient="records")
    }


Instrumentul pentru recomandari:

In [35]:
def tool_get_reorder_recommendations(customer_id):
    customer_exists = (
        df_clean["CustomerID"] == float(customer_id)
    ).any()

    if not customer_exists:
        return {
            "success": False,
            "message": f"Clientul {customer_id} nu exista."
        }

    recommendations = (
        get_reorder_recommendations_for_agent(customer_id)
    )

    return {
        "success": True,
        "customer_id": int(customer_id),
        "recommendations": recommendations,
        "stock_is_simulated": True
    }

Instrumentul pentru obtinerea detaliilor unui produs

In [36]:
def tool_get_product_details(stock_code):
    code = str(stock_code).strip()

    product_data = df_clean[
        df_clean["StockCode"].astype(str).str.strip() == code
    ]

    if product_data.empty:
        return {
            "success": False,
            "message": f"Produsul cu codul {code} nu exista."
        }

    return {
        "success": True,
        "stock_code": code,
        "description": product_data["Description"].mode().iloc[0],
        "total_quantity_sold": int(
            product_data["Quantity"].sum()
        ),
        "number_of_invoices": int(
            product_data["InvoiceNo"].nunique()
        ),
        "number_of_customers": int(
            product_data["CustomerID"].nunique()
        )
    }

Intregistrez instrumentele create pentru agent

In [37]:
TOOLS = {
    "get_order_history": tool_get_order_history,
    "get_reorder_recommendations": tool_get_reorder_recommendations,
    "get_product_details": tool_get_product_details
}

print("Instrumente disponibile:")
for tool_name in TOOLS:
    print("-", tool_name)

Instrumente disponibile:
- get_order_history
- get_reorder_recommendations
- get_product_details


Construiesc mecanismul de selectare a functiei necesare si apelarea ei

In [38]:
import re

def run_agent_tool(user_message):
    message = user_message.lower()

    #cauta id-ul in mesajul trimis de user
    ids = re.findall(r"\b\d{5}\b", user_message)

    #cererile despre produs
    product_keywords = [
        "detalii produs",
        "informatii despre produs",
        "despre produs",
        "codul produsului",
        "produsul cu codul"
    ]

    if any(keyword in message for keyword in product_keywords):
        if not ids:
            return {
                "success": False,
                "message": "Specificati codul produsului."
            }

        tool_name = "get_product_details"
        arguments = {"stock_code": ids[-1]}

    #cereri despre istoric
    elif any(keyword in message for keyword in [
        "istoric", "ce a cumparat", "ce a comandat",
        "istoricul comenzilor"
    ]):
        if not ids:
            return {
                "success": False,
                "message": "Specificati ID-ul clientului."
            }

        tool_name = "get_order_history"
        arguments = {"customer_id": int(ids[-1])}

    #cereri pentru recumparare
    elif any(keyword in message for keyword in [
        "reaprovizionare", "recomandari", "ce produse sa comand"
    ]):
        if not ids:
            return {
                "success": False,
                "message": "Specificati ID-ul clientului."
            }

        tool_name = "get_reorder_recommendations"
        arguments = {"customer_id": int(ids[-1])}

    else:
        return {
            "success": False,
            "message": "Nu am identificat tipul cererii."
        }

    #apeleaza doar tool-urile facute
    result = TOOLS[tool_name](**arguments)

    return {
        "tool_used": tool_name,
        "arguments": arguments,
        "result": result
    }

Pana acum, agentul a ales instructiunea prin cuvintele cheie. Acum modific astfel incat agentul sa interpreteze cererea si sa aleaga instrumentul, dar validarea cererii va fi facuta prin python.

In [41]:
import json
import re

TOOL_DESCRIPTIONS = """
Instrumente disponibile:

1. get_order_history
   Parametru: customer_id (integer)
   Foloseste-l cand utilizatorul intreaba despre comenzile
   sau produsele cumparate de un client.

2. get_reorder_recommendations
   Parametru: customer_id (integer)
   Foloseste-l cand utilizatorul cere recomandari
   de reaprovizionare pentru un client.

3. get_product_details
   Parametru: stock_code (string)
   Foloseste-l cand utilizatorul cere informatii
   despre un anumit produs.
"""

def select_tool_with_llm(user_message):
    messages = [
        {
            "role": "system",
            "content": (
                "Esti un router pentru un agent AI de business. "
                "Alege un singur instrument din lista disponibila. "
                "Nu raspunde la intrebarea utilizatorului. "
                "Nu inventa ID-uri. "
                "Returneaza exclusiv un obiect JSON valid, "
                "fara Markdown, cu formatul: "
                '{"tool": "nume_instrument", "arguments": {}}. '
                'Daca nu poti determina instrumentul sau argumentele, '
                'returneaza {"tool": "none", "arguments": {}}.'
                "\n\n" + TOOL_DESCRIPTIONS
            )
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    #extrage din json
    match = re.search(r"\{.*\}", generated, re.DOTALL)

    if not match:
        return {"tool": "none", "arguments": {}}

    try:
        selection = json.loads(match.group())
    except json.JSONDecodeError:
        return {"tool": "none", "arguments": {}}

    return selection

Validez alegerile facute de agent

In [42]:
import re

def validate_user_request(message):
    text = message.lower().strip()
    greetings = [
        "buna", "buna ziua", "salut", "hello", "hei",
        "ce poti face", "cu ce ma poti ajuta",
        "ce stii sa faci", "ajutor"
    ]

    if any(greeting in text for greeting in greetings):
        return {
            "valid": True,
            "tool": "greeting",
            "arguments": {}
        }


    #intentia
    product_keywords = [
        "detalii produs",
        "informatii despre produs",
        "informatii produs",
        "codul produsului",
        "despre produs",
        "detalii produsului"
    ]

    history_keywords = [
        "istoric",
        "ce a cumparat",
        "ce a comandat",
        "istoricul comenzilor"
    ]

    reorder_keywords = [
        "reaprovizion",
        "recomandari",
        "ce produse sa comand",
        "ce produse ar trebui comandate"
    ]

    if any(keyword in text for keyword in product_keywords):
        intent = "get_product_details"

    elif any(keyword in text for keyword in history_keywords):
        intent = "get_order_history"

    elif any(keyword in text for keyword in reorder_keywords):
        intent = "get_reorder_recommendations"

    else:
        return {
            "valid": False,
            "message": (
                "Nu am identificat cererea. Poti solicita "
                "istoricul unui client, recomandari de "
                "reaprovizionare sau detalii despre un produs."
            )
        }

    #extrage id ul clientului din mesaj
    ids = re.findall(r"\b\d{5}\b", message)

    if not ids:
        return {
            "valid": False,
            "message": (
                "Nu am identificat ID-ul necesar. "
                "Te rog sa specifici ID-ul clientului "
                "sau codul produsului."
            )
        }

    #alege identificatorul potrivit intentiei
    identifier = ids[-1]

    if intent == "get_product_details":
        arguments = {"stock_code": identifier}
    else:
        arguments = {"customer_id": int(identifier)}

    return {
        "valid": True,
        "tool": intent,
        "arguments": arguments
    }


Construiesc agentul complet

In [40]:
def run_llm_agent(user_message):
    #valideaza cererea utilizatorului
    validation = validate_user_request(user_message)

    if not validation["valid"]:
        return {
            "success": False,
            "message": validation["message"]
        }
    if validation["tool"] == "greeting":
        return {
            "tool_used": "greeting",
            "arguments": {},
            "result": {
                "success": True,
                "message": (
                    "Buna! Sunt AI Reordering Assistant.\n\n"
                    "Te pot ajuta cu:\n"
                    "- istoricul comenzilor unui client;\n"
                    "- recomandari de reaprovizionare;\n"
                    "- detalii despre un produs.\n\n"
                    "Exemple:\n"
                    "- Ce a cumparat clientul 14911?\n"
                    "- Ce produse ar trebui reaprovizionate pentru clientul 14911?\n"
                    "- Arata detalii produsului cu codul 23169."
                )
            }
        }

    #cere llm-ului selectia tool-ului care trebuie folosit
    selection = select_tool_with_llm(user_message)

    expected_tool = validation["tool"]
    selected_tool = selection.get("tool")

    #verific daca llm ul a ales tool-ul corect
    if selected_tool != expected_tool:
        return {
            "success": False,
            "message": (
                "Nu am putut confirma actiunea potrivita. "
                "Te rog sa reformulezi cererea."
            )
        }

    #foloseste argumentele validate, nu ID-uri neverificate
    arguments = validation["arguments"]

    #executa tool-ul
    result = TOOLS[expected_tool](**arguments)

    return {
        "tool_used": expected_tool,
        "arguments": arguments,
        "result": result
    }


Acum ca agentul stie sa afiseze informatia din fisierul JSON, trebuie invatat sa afiseze informatia ca o conversatie.

In [44]:
def format_agent_response(agent_output):
    if agent_output.get("tool_used") == "greeting":
      return agent_output["result"]["message"]

    if not agent_output.get("success", True):
        return agent_output.get(
            "message",
            "Nu am putut procesa cererea."
        )

    tool_name = agent_output["tool_used"]
    result = agent_output["result"]

    if not result.get("success", False):
        return result.get(
            "message",
            "Nu am putut obtine rezultatul."
        )

    if tool_name == "get_order_history":
        orders = result["orders"]

        lines = [
            f"Istoricul recent al clientului {result['customer_id']}:"
        ]

        for order in orders:
            lines.append(
                f"- {order['Description'].strip()} "
                f"(cod {order['StockCode']}), "
                f"cantitate: {order['Quantity']}, "
                f"pret unitar: {order['UnitPrice']:.2f}"
            )

        lines.append(
            "\nSunt afisate ultimele 10 pozitii din istoric, "
            "nu neaparat 10 comenzi distincte."
        )
        return "\n".join(lines)

    elif tool_name == "get_reorder_recommendations":
        recommendations = result["recommendations"]

        if not recommendations:
            return (
                "Nu am gasit produse eligibile pentru "
                "reaprovizionare pentru acest client."
            )

        lines = [
            f"Recomandari pentru clientul {result['customer_id']}:"
        ]

        for item in recommendations:
            lines.append(
                f"- {item['Description'].strip()} "
                f"(cod {item['StockCode']}): "
                f"cantitate propusa {item['suggested_quantity']} "
                f"bucati; stoc simulat "
                f"{item['stock_available']} bucati."
            )

        lines.append(
            "\nAtentie: stocurile sunt simulate, iar "
            "recomandarile sunt estimari bazate pe istoricul "
            "de cumparare. Nu s-a plasat nicio comanda."
        )
        return "\n".join(lines)

    elif tool_name == "get_product_details":
        return (
            f"Detalii produs: {result['description'].strip()}\n"
            f"Cod: {result['stock_code']}\n"
            f"Cantitate totala vanduta in dataset: "
            f"{result['total_quantity_sold']}\n"
            f"Numar de facturi: {result['number_of_invoices']}\n"
            f"Numar de clienti: {result['number_of_customers']}"
        )

    return "Instrumentul a fost executat, dar nu exista un format de afisare."


Test

In [45]:
test_messages = [
    "Ce a cumparat clientul 14911?",
    "Ce produse ar trebui reaprovizionate pentru clientul 14911?",
    "Arata detalii produsului cu codul 23169.",
    "Ce a cumparat clientul?",
    "Ce a cumparat clientul 99999?"
]

for message in test_messages:
    print("\n" + "=" * 50)
    print("Cerere:", message)
    output = run_llm_agent(message)

    if output.get("success") is False:
        print("Raspuns:", output["message"])
    else:
        print("Instrument:", output["tool_used"])
        print(format_agent_response(output))



Cerere: Ce a cumparat clientul 14911?
Instrument: get_order_history
Istoricul recent al clientului 14911:
- CERAMIC LOVE HEART MONEY BANK (cod 22647), cantitate: 36, pret unitar: 0.39
- SEWING BOX RETROSPOT DESIGN (cod 22509), cantitate: 4, pret unitar: 3.95
- TV DINNER TRAY DOLLY GIRL (cod 22472), cantitate: 8, pret unitar: 1.95
- GROW YOUR OWN FLOWERS SET OF 3 (cod 22442), cantitate: 12, pret unitar: 1.95
- PAPERWEIGHT VINTAGE PAISLEY (cod 22395), cantitate: 12, pret unitar: 0.39
- MONEY BOX POCKET MONEY DESIGN (cod 22405), cantitate: 12, pret unitar: 0.39
- MONEY BOX HOUSEKEEPING DESIGN (cod 22410), cantitate: 12, pret unitar: 0.39
- PAPERWEIGHT VINTAGE COLLAGE (cod 22393), cantitate: 24, pret unitar: 0.39
- HOT STUFF HOT WATER BOTTLE (cod 23358), cantitate: 18, pret unitar: 1.25
- ART LIGHTS,FUNK MONKEY (cod 79302M), cantitate: 6, pret unitar: 2.95

Sunt afisate ultimele 10 pozitii din istoric, nu neaparat 10 comenzi distincte.

Cerere: Ce produse ar trebui reaprovizionate pentru 

Voi introduce o mica interfata pentru a comunica cu agentul

Pentru asta voi instala Gradio

In [46]:
!pip install -q gradio

Construiesc raspunsurile agentului

In [47]:
import gradio as gr

def chat_with_agent(message, history):
    message = message.strip().lower()

    if not message:
        return "Scrie o intrebare, te rog."
    greetings = [
        "buna", "buna ziua", "salut", "hello", "hei",
        "ce poti face", "cu ce ma poti ajuta",
        "ce stii sa faci", "ajutor"
    ]

    if any(greeting in message for greeting in greetings):
        return (
            "Buna! Sunt AI Reordering Assistant.\n\n"
            "Te pot ajuta cu:\n"
            "- istoricul comenzilor unui client;\n"
            "- recomandari de reaprovizionare;\n"
            "- detalii despre un produs.\n\n"
            "Exemple:\n"
            "- Ce a cumparat clientul 14911?\n"
            "- Ce produse ar trebui reaprovizionate pentru clientul 14911?\n"
            "- Arata detalii produsului cu codul 23169."
        )

    try:
        output = run_llm_agent(message)

        if not output.get("result", {}).get("success", True):
            return output["result"].get(
                "message",
                "Nu am putut obtine datele solicitate."
            )

        return format_agent_response(output)

    except Exception as e:
        print("Eroare:", repr(e))
        return "A aparut o eroare. Te rog sa reformulezi cererea."


Fac mai multe teste de validare a agentului

In [48]:
test_cases = [
    "Ce a cumparat clientul 14911?",
    "Ce produse ar trebui reaprovizionate pentru clientul 14911?",
    "Arata detalii produsului cu codul 23169.",
    "Ce a cumparat clientul?",
    "Ce a cumparat clientul 99999?",
    "Buna, ce poti face?"
]

for question in test_cases:
    print("\n" + "=" * 60)
    print("Cerere:", question)

    output = run_llm_agent(question)
    print(format_agent_response(output))



Cerere: Ce a cumparat clientul 14911?
Istoricul recent al clientului 14911:
- CERAMIC LOVE HEART MONEY BANK (cod 22647), cantitate: 36, pret unitar: 0.39
- SEWING BOX RETROSPOT DESIGN (cod 22509), cantitate: 4, pret unitar: 3.95
- TV DINNER TRAY DOLLY GIRL (cod 22472), cantitate: 8, pret unitar: 1.95
- GROW YOUR OWN FLOWERS SET OF 3 (cod 22442), cantitate: 12, pret unitar: 1.95
- PAPERWEIGHT VINTAGE PAISLEY (cod 22395), cantitate: 12, pret unitar: 0.39
- MONEY BOX POCKET MONEY DESIGN (cod 22405), cantitate: 12, pret unitar: 0.39
- MONEY BOX HOUSEKEEPING DESIGN (cod 22410), cantitate: 12, pret unitar: 0.39
- PAPERWEIGHT VINTAGE COLLAGE (cod 22393), cantitate: 24, pret unitar: 0.39
- HOT STUFF HOT WATER BOTTLE (cod 23358), cantitate: 18, pret unitar: 1.25
- ART LIGHTS,FUNK MONKEY (cod 79302M), cantitate: 6, pret unitar: 2.95

Sunt afisate ultimele 10 pozitii din istoric, nu neaparat 10 comenzi distincte.

Cerere: Ce produse ar trebui reaprovizionate pentru clientul 14911?
Recomandari pe

Test in interfata:

In [49]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chat_with_agent,
    title="AI Reordering Assistant",
    description=(
        "Asistent pentru istoricul comenzilor, recomandari "
        "de reaprovizionare si detalii despre produse. "
        "Nu plaseaza comenzi."
    ),
    examples=[
        "Ce a cumparat clientul 14911?",
        "Ce produse ar trebui reaprovizionate pentru clientul 14911?",
        "Arata detalii produsului cu codul 23169."
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b9d05d1d09417893a2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
